# **02 웹 검색 에이전트 구현하기**

### 학습 내용
1. Tavily Search API 이해 및 설정
2. TavilySearch 도구 사용법
3. 웹 검색 결과 처리
4. create_agent와 Tavily Search 통합
5. 다중 도구 에이전트 구성

## 1. 환경 설정

- OpenAI API Key 발급: https://platform.openai.com/api-keys
- Tavily API Key 발급: https://app.tavily.com/home

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

if os.environ.get("OPENAI_API_KEY"):
    print("OPENAI API Key가 설정되었습니다.")

if os.getenv("TAVILY_API_KEY"):
    print("TAVILY API Key가 설정되었습니다.")

OPENAI API Key가 설정되었습니다.
TAVILY API Key가 설정되었습니다.


## 2. Tavily Search API 소개

**Tavily Search API**는 AI 에이전트(LLM)를 위해 특별히 설계된 검색 엔진입니다.

### 주요 특징

- **실시간 검색**: 최신 정보를 빠르게 검색
- **AI 최적화**: LLM이 이해하기 쉬운 형식으로 결과 제공
- **정확한 결과**: 신뢰할 수 있는 소스에서 팩트 기반 정보 제공
- **무료 티어**: 월 1,000회 무료 검색 제공

## 3. TavilySearch 도구 기본 사용법

[TavilySearch](https://reference.langchain.com/python/langchain-tavily/tavily_search/TavilySearch) 도구를 직접 사용하여 웹 검색을 수행해봅시다.

In [3]:
from langchain_tavily import TavilySearch

# 기본 설정으로 TavilySearch 도구 생성
search_tool = TavilySearch(
    max_results=3,  # 최대 검색 결과 수
    topic="general",  # 검색 카테고리: "general", "news", "finance"
)

print(f"도구 이름: {search_tool.name}")
print(f"도구 설명: {search_tool.description}")

도구 이름: tavily_search
도구 설명: A search engine optimized for comprehensive, accurate, and trusted results. Useful for when you need to answer questions about current events. It not only retrieves URLs and snippets, but offers advanced search depths, domain management, time range filters, and image search, this tool delivers real-time, accurate, and citation-backed results.Input should be a search query.


In [4]:
result = search_tool.invoke({"query": "LangChain이란 무엇인가요?"})

print("\n=== 검색 결과 ===")
print(result)


=== 검색 결과 ===
{'query': 'LangChain이란 무엇인가요?', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://www.samsungsds.com/kr/insights/what-is-langchain.html', 'title': '랭체인 LangChain 이란 무엇인가? | 인사이트리포트 | 삼성SDS', 'content': '**LangChain(랭체인)은 대규모 언어 모델(LLM)을 활용한 애플리케이션 개발에 특화된 오픈소스 프레임워크입니다.** LangChain은 기존 언어 모델의 한계를 극복하고, AI 기술을 활용한 새로운 애플리케이션을 구축할 수 있는 중요한 도구로 자리 잡고 있습니다. LLM의 잠재력을 극대화하기 위해 데이터베이스, 파일 시스템 등과 같은 다양한 데이터 소스와의 통합을 지원하여, 실시간 데이터와 상호작용하는 애플리케이션을 구축할 수 있습니다. LangChain은 다음과 같은 기능들을 통해 다양한 산업에서 혁신적인 애플리케이션을 구축할 수 있는 강력한 도구로 자리 잡고 있습니다. LangChain은 다양한 언어 모델과 쉽게 통합할 수 있으며, 이를 통해 복잡하고 다양한 애플리케이션을 구축할 수 있습니다. 또한, LangChain은 여러 언어 모델과의 통합이 가능하여, 다양한 사용 사례에 맞춘 애플리케이션을 쉽게 구축할 수 있습니다. * **실시간 데이터 통합 :**\xa0API를 통해 실시간 데이터를 언어 모델에 제공하여, 사용자가 요청할 때마다 최신 정보를 기반으로 응답을 생성할 수 있습니다. * **데이터 반응형 애플리케이션 :**\xa0LangChain을 통해 구축된 애플리케이션은 실시간 데이터와 지속적으로 상호작용할 수 있으며, 이를 통해 사용자가 필요한 순간에 정확하고 관련성 높은 정보를 제공받을 수 있습니다. 이 시스템은 금융 전문가들이 복잡한 질문에 대해 정확한 답변을 얻을 수 있도록 도와주며, 

In [5]:
result["results"]

[{'url': 'https://www.samsungsds.com/kr/insights/what-is-langchain.html',
  'title': '랭체인 LangChain 이란 무엇인가? | 인사이트리포트 | 삼성SDS',
  'content': '**LangChain(랭체인)은 대규모 언어 모델(LLM)을 활용한 애플리케이션 개발에 특화된 오픈소스 프레임워크입니다.** LangChain은 기존 언어 모델의 한계를 극복하고, AI 기술을 활용한 새로운 애플리케이션을 구축할 수 있는 중요한 도구로 자리 잡고 있습니다. LLM의 잠재력을 극대화하기 위해 데이터베이스, 파일 시스템 등과 같은 다양한 데이터 소스와의 통합을 지원하여, 실시간 데이터와 상호작용하는 애플리케이션을 구축할 수 있습니다. LangChain은 다음과 같은 기능들을 통해 다양한 산업에서 혁신적인 애플리케이션을 구축할 수 있는 강력한 도구로 자리 잡고 있습니다. LangChain은 다양한 언어 모델과 쉽게 통합할 수 있으며, 이를 통해 복잡하고 다양한 애플리케이션을 구축할 수 있습니다. 또한, LangChain은 여러 언어 모델과의 통합이 가능하여, 다양한 사용 사례에 맞춘 애플리케이션을 쉽게 구축할 수 있습니다. * **실시간 데이터 통합 :**\xa0API를 통해 실시간 데이터를 언어 모델에 제공하여, 사용자가 요청할 때마다 최신 정보를 기반으로 응답을 생성할 수 있습니다. * **데이터 반응형 애플리케이션 :**\xa0LangChain을 통해 구축된 애플리케이션은 실시간 데이터와 지속적으로 상호작용할 수 있으며, 이를 통해 사용자가 필요한 순간에 정확하고 관련성 높은 정보를 제공받을 수 있습니다. 이 시스템은 금융 전문가들이 복잡한 질문에 대해 정확한 답변을 얻을 수 있도록 도와주며, LangChain의 실시간 데이터 통합과 맞춤형 프롬프팅 기능을 효과적으로 활용하고 있습니다.',
  'score': 0.9999968,
  'raw_content': None},
 {'url': 'ht

### 주요 파라미터

| 파라미터 | 설명 | 기본값 |
|---------|------|-------|
| `max_results` | 반환할 최대 검색 결과 수 | 5 |
| `topic` | 검색 카테고리 ("general", "news", "finance") | "general" |
| `include_answer` | 질문에 대한 직접 답변 포함 여부 | False |
| `include_raw_content` | 전체 HTML 콘텐츠 포함 여부 | False |
| `include_images` | 관련 이미지 포함 여부 | False |
| `search_depth` | 검색 깊이 ("basic", "advanced") | "basic" |
| `include_domains` | 포함할 도메인 리스트 | None |
| `exclude_domains` | 제외할 도메인 리스트 | None |
| `time_range` | 시간 범위 ("day", "week", "month", "year") | None |

## 4. 특정 도메인으로 검색 제한하기

특정 웹사이트만 검색하거나 특정 사이트를 제외할 수 있습니다.

In [6]:
# Wikipedia만 검색하기
wiki_search = TavilySearch(
    max_results=3,
    include_domains=["wikipedia.org"],  # Wikipedia만 검색
)

result = wiki_search.invoke({"query": "인공지능"})

print("\n=== Wikipedia 검색 결과 ===")
print(result)


=== Wikipedia 검색 결과 ===
{'query': '인공지능', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://ko.wikipedia.org/wiki/%EC%9D%B8%EA%B3%B5%EC%A7%80%EB%8A%A5', 'title': '인공지능 - 위키백과, 우리 모두의 백과사전', 'content': '인공지능(人工智能, 영어: artificial intelligence, AI )은 인간의 학습능력, 추론능력, 지각능력을 인공적으로 구현하려는 컴퓨터 과학의 세부분야 중 하나이다.', 'score': 0.834852, 'raw_content': None}, {'url': 'https://ko.wikipedia.org/wiki/%EC%9D%B8%EA%B3%B5%EC%A7%80%EB%8A%A5%EC%9D%98_%EA%B0%9C%EC%9A%94', 'title': '인공지능의 개요 - 위키백과, 우리 모두의 백과사전', 'content': '인공지능(Artificial intelligence, AI)이라 함은 기계나 소프트웨어에 의해 표출되는 지능이다. 지능적 행위를 할 수 있는 컴퓨터와 컴퓨터 소프트웨어를 만드는 방법을', 'score': 0.8036831, 'raw_content': None}, {'url': 'https://ko.wikipedia.org/wiki/%EC%9D%B8%EA%B3%B5%EC%A7%80%EB%8A%A5%EC%9D%98_%EC%9D%91%EC%9A%A9', 'title': '인공지능의 응용 - 위키백과, 우리 모두의 백과사전', 'content': '인공지능은 학습, 추론, 문제 해결, 인식, 의사 결정과 같이 인간 지능과 일반적으로 관련된 작업을 수행하는 전산 시스템의 능력이다. 인공지능(AI)은 산업 및 학계 전반', 'score': 0.78944516, 'raw_content': None}], 'respons

In [7]:
# 특정 도메인 제외하기
filtered_search = TavilySearch(
    max_results=3,
    exclude_domains=["reddit.com", "twitter.com", "blog.naver.com", "tistory.com"],
)

result = filtered_search.invoke({"query": "파이썬 프로그래밍 팁"})

print("\n=== 필터링된 검색 결과 ===")
print(result)


=== 필터링된 검색 결과 ===
{'query': '파이썬 프로그래밍 팁', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://kr.linkedin.com/pulse/10-tips-ideas-python-beginners-expert-journey-ayush-gupta-pfljf?tl=ko', 'title': '파이썬 초보자부터 전문가까지 여정을 위한 10가지 팁과 아이디어', 'content': '웹 개발을 위해 파이썬을 배우고 싶다면 Django나 Flask 같은 프레임워크에 집중하는 게 좋을 것 같아요. 목표를 알면 자신의 필요와 관심사에 맞게 학습', 'score': 0.9914225, 'raw_content': None}, {'url': 'https://yozm.wishket.com/magazine/detail/1605/', 'title': '파이썬 초보자가 저지르는 10가지 실수 - 요즘IT', 'content': "1. import *을 사용함 · 2. 예외 처리: 'except' 절에 예외를 지정하지 않음 · 3. 수학 계산에 Numpy를 사용하지 않음 · 4. 이전에 열었던 파일을 닫지 않음.", 'score': 0.9314625, 'raw_content': None}, {'url': 'https://wikidocs.net/84381', 'title': '1.1 파이썬 - 실용 파이썬 프로그래밍 - 위키독스', 'content': '파이썬을 처음으로 배운다면 이와 같이 "천천히" 학습하는 것을 권장한다. 속도를 늦추고 직접 입력하면서 언어를 음미하고, 무엇을 하고 있는지 스스로 생각할 시간을', 'score': 0.9309621, 'raw_content': None}], 'response_time': 0.82, 'request_id': '987e059c-2305-4601-a067-69787eb6396b'}


## 5. create_agent와 Tavily Search 통합

이제 Tavily Search를 에이전트와 통합하여 더 지능적인 검색 에이전트를 만들어봅시다.

In [8]:
from langchain.agents import create_agent

# Tavily Search 도구 생성
tavily_tool = TavilySearch(
    max_results=5,
    topic="general",
)

# 검색 에이전트 생성
search_agent = create_agent(
    model="gpt-4o-mini",
    tools=[tavily_tool],
    system_prompt="""당신은 유용한 연구 보조 AI입니다.
    Tavily 검색 도구를 사용하여 정확하고 최신 정보를 찾아주세요.""",
)

In [9]:
for chunk_msg, metadata in search_agent.stream({"messages": [{"role": "user", "content": "2026년의 AI 트렌드는?"}]}, stream_mode="messages"):
    print(chunk_msg.content, end="", flush=True)

{"query": "AI trends 2026", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://sloanreview.mit.edu/article/five-trends-in-ai-and-data-science-for-2026/", "title": "Five Trends in AI and Data Science for 2026", "content": "Share\n\nTwitter Facebook Linkedin  Copy Link\n\nSummary: \n\nMIT SMR columnists Thomas H. Davenport and Randy Bean see five AI trends to pay attention to in 2026: deflation of the AI bubble and subsequent hits to the economy; growth of the “factory” infrastructure for all-in AI adapters; greater focus on generative AI as an organizational resource rather than an individual one; continued progression toward value from agentic AI, despite the hype; and ongoing questions around who should manage data and AI. [...] Last year, like virtually everyone else, we predicted that agentic AI would be on the rise. Although we acknowledged that the technology was being hyped and had some challenges, we underestimated the degree of both. Agents 

## 6. 실전 연습 1: 뉴스 분석 에이전트

최신 뉴스를 검색하고 분석하는 에이전트를 만들어봅시다.

In [10]:
# 뉴스 검색용 도구
news_search = TavilySearch(
    max_results=5,
    topic="news",
    start_date="2025-01-01"
)

from datetime import datetime
current_date = datetime.now().strftime("%Y-%m-%d")

# 뉴스 분석 에이전트
news_agent = create_agent(
    model="gpt-4o-mini",
    tools=[news_search],
    system_prompt=f"""당신은 뉴스 분석 보조 AI입니다.
    시사 이슈에 대해 질문을 받으면:
    1. 해당 주제에 대한 최신 뉴스 검색
    2. 출처 제공

    분석은 객관적이고 사실에 기반해야 합니다.
    현재 날짜 : {current_date}""",
)

In [11]:
for chunk_msg, metadata in news_agent.stream({"messages": [{"role": "user", "content": "2026년의 밀라노 올림픽 결과"}]}, stream_mode="messages"):
    print(chunk_msg.content, end="", flush=True)

{"query": "2026 Milan Olympics results", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://apnews.com/article/milan-verona-juventus-serie-a-ba573d5c6c4d4f4a6813061e8a8dd300", "title": "AC Milan steadies Champions League push with 1-0 win at last-place Verona - AP News", "score": 0.3097232, "published_date": "Sun, 19 Apr 2026 15:15:35 GMT", "content": "Test Your News I.Q. 2026 Elections Election Results Election calendar White House Congress Supreme Court The latest AP-NORC polls Ground Game. Test Your News I.Q. 2026 Elections Election Results Election calendar White House Congress Supreme Court The latest AP-NORC polls Ground Game. More than half the world’s population sees AP journalism every day. AC Milan’s Adrien Rabiot, left, celebrates scoring during the Serie A soccer match between Hellas Verona and A.C. Milan in Verona, Italy, Sunday April 19, 2026. AC Milan’s Adrien Rabiot, left, celebrates scoring during the Serie A soccer match between He

In [12]:
for chunk_msg, metadata in news_agent.stream({"messages": [{"role": "user", "content": "이란 전쟁으로인한 원유 가격 급등"}]}, stream_mode="messages"):
    print(chunk_msg.content, end="", flush=True)

{"query": "Iran war oil price surge", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.utilitydive.com/news/iran-war-oil-price-construction-stress/817446/", "title": "Iran war impacts on oil prices spiked construction stress, increased abandonments - Utility Dive", "score": 0.99986553, "published_date": "Tue, 14 Apr 2026 09:23:24 GMT", "content": "# Iran war impacts on oil prices spiked construction stress, increased abandonments. Construction saw a 22.8% surge in project abandonments month over month in March, which ConstructConnect linked to economic impacts brought on by the war. Construction stress snapped back up in March after one of its lowest readings in more than a year, according to the latest data from Cincinnati-based ConstructConnect. The Project Stress Index, a measure of construction projects that have been paused, abandoned or exhibit a delayed bid date, increased 4.2% month over month in March. A 22.8% surge in project abandon

## 7. 실전 연습 2: 학술 연구 보조 에이전트

특정 도메인(학술 사이트)만 검색하는 연구 보조 에이전트를 만들어봅시다.

In [13]:
# 학술 검색용 도구
academic_search = TavilySearch(
    max_results=5,
    include_domains=[
        "arxiv.org",
        "scholar.google.com",
        "wikipedia.org",
        "nature.com",
        "sciencedirect.com"
    ],
    search_depth="advanced",
)

# 연구 보조 에이전트
research_agent = create_agent(
    model="gpt-4o-mini",
    tools=[academic_search],
    system_prompt="""당신은 학술 연구 보조 AI입니다.
    사용자가 신뢰할 수 있는 출처에서 학술 정보를 찾도록 도와주세요.
    다음에 중점을 두세요:
    - 과학적 정확성
    - 동료 평가된 출처
    - 최신 연구 결과
    - 복잡한 주제에 대한 명확한 설명

    추가 읽기를 위해 항상 출처 URL을 제공하세요.""",
)

In [17]:
from langchain_core.messages import AIMessageChunk

for chunk_msg, metadata in research_agent.stream({"messages": [{"role": "user", "content": "2025년 ~ 2026년의 프롬프트 엔지니어링의 연구 동향을 알려주세요."}]}, stream_mode="messages"):
    if type(chunk_msg) == AIMessageChunk:
        print(chunk_msg.content, end="", flush=True)

2025년과 2026년의 프롬프트 엔지니어링 연구 동향은 다음과 같습니다:

1. **맥락 공학(Context Engineering)**: 2025년부터 프롬프트 엔지니어링의 개념이 발전하여 맥락 공학이라는 새로운 분야가 등장했습니다. 이는 사용자가 정보를 어떻게 요청하고, 모델이 행동할 때 어떤 맥락을 알고, 보유하고 있는지를 포함하는 시스템 공학으로 발전했습니다. 맥락 공학은 이제 더 이상 단순한 프롬프트 작성 기술이 아니라, 모델의 상태를 설계하는 데 필요한 필수적인 기술로 자리잡았습니다 [출처: arXiv](https://arxiv.org/pdf/2603.09619).

2. **자동화된 프롬프트 엔지니어링 파이프라인**: 2026년에는 자동화된 롤 기반 프롬프트 엔지니어링 시스템이 개발되어, 대규모 언어 모델(LLMs)의 응답 정확도를 향상시키는 데 크게 기여했습니다. 이 시스템은 사용자 맞춤형 프롬프트 구성을 단순화하여 여러 데이터셋에서 정확도가 최대 97.6%까지 향상되었습니다 [출처: ScienceDirect](https://www.sciencedirect.com/science/article/abs/pii/S0957417425033044).

3. **교육적 적용**: 프롬프트 엔지니어링은 교육 분야에도 적용되어, 예비 교사들이 GenAI를 활용하여 적응형 수업 계획을 공동 구성하는 데 중요한 역할을 하고 있습니다. 이 연구는 효과적인 프롬프트 엔지니어링이 교사의 전문 지식과 밀접하게 연관되어 있음을 보여주며, AI 특화 교육이 교사 교육에 필수적임을 강조합니다 [출처: ScienceDirect](https://www.sciencedirect.com/science/article/pii/S0360131525002532).

4. **코드 요약의 시스템적 문헌 검토**: 프롬프트 엔지니어링 기술과 벤치마크를 종합적으로 검토한 연구가 진행되어, LLM 기반의 코드 요약이 더 신뢰 가능하고 효과적이 되도록 하는 프롬프트의 설계 및 최적화에 초점을 맞추고

In [18]:
for chunk_msg, metadata in research_agent.stream({"messages": [{"role": "user", "content": "코딩 특화 AI Agent에 대한 연구 자료를 찾아주세요."}]}, stream_mode="messages"):
    print(chunk_msg.content, end="", flush=True)

{"query": "coding specialized AI agent research", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://arxiv.org/html/2604.02544v1", "title": "Developer Experience with AI Coding Agents: HTTP Behavioral ...", "content": "The rapid adoption of AI coding agents and AI assistant web services is fundamentally changing how developers discover, consume, and interact with technical documentation. This paper studies that transformation across three interconnected dimensions: documentation accessibility, content analytics, and feedback systems. We present an empirical study of HTTP request fingerprints from nine AI coding agents (Aider, Antigravity, Claude Code, Cline, Cursor, Junie, OpenCode, VS Code, and Windsurf) and six AI assistant services (ChatGPT, Claude, Google Gemini, Google NotebookLM, MistralAI, and Perplexity) accessing a live developer documentation endpoint, revealing identifiable behavioral signatures in HTTP runtime environments, pre-fetch str

## 12. 실습 과제

### 📖 과제 1: 도메인 특화 웹 검색 에이전트 만들기

위에서 학습한 내용을 바탕으로, **특정 도메인에 특화된 웹 검색 에이전트**를 만들어보세요.

- 출처를 포함한 정확한 답변 제공

In [15]:
# CODE HERE

<details>
<summary>참고 예시 (클릭하여 펼치기)</summary>

### 레시피 검색 에이전트 예시

```python
from langchain_tavily import TavilySearch
from langchain.agents import create_agent

# 레시피 전문 검색 도구
recipe_search = TavilySearch(
    max_results=5,
    include_domains=[
        "allrecipes.com",
        "foodnetwork.com", 
        "tasty.co",
        "bbcgoodfood.com",
        "10000recipe.com"  # 만개의 레시피
    ],
)

# 레시피 에이전트
recipe_agent = create_agent(
    model="gpt-4o-mini",
    tools=[recipe_search],
    system_prompt="""당신은 친절한 요리 도우미 AI입니다.
    사용자가 요리 레시피를 요청하면:
    1. 신뢰할 수 있는 레시피 사이트에서 최신 정보 검색
    2. 재료, 조리 시간, 난이도를 포함하여 요약
    3. 초보자도 이해하기 쉽게 설명
    4. 출처 URL을 반드시 제공하여 자세한 레시피 확인 가능하도록 함
    
    항상 맛있고 건강한 요리를 만들 수 있도록 도와주세요!""",
)

# 테스트
for chunk_msg, metadata in recipe_agent.stream(
    {"messages": [{"role": "user", "content": "초보자를 위한 간단한 파스타 레시피 알려줘"}]},
    stream_mode="messages"
):
    print(chunk_msg.content, end="", flush=True)
```

</details>

---
### 참고 자료

- [Tavily Search Integration](https://docs.langchain.com/oss/python/integrations/tools/tavily_search)
- [Tavily API Documentation](https://docs.tavily.com/documentation/api-reference/endpoint/search)
- [LangChain Agents](https://docs.langchain.com/oss/python/langchain/agents)
- [LangChain Tools](https://docs.langchain.com/oss/python/langchain/tools)